Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Train

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]
PROTOCOL1_TRAIN_INDICES = {
    1: [1, 2],
    2: [1, 2],
    3: [1, 2],
    4: [1, 2]
}

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print("🔧 Denoising image...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO FUSE FINGERS FOR TRAINING ===
def fuse_fingers_p1(subject_path, subject_id):
    fused_samples = []
    labels = []

    sample_counter = 0

    for img_num, aug_list in PROTOCOL1_TRAIN_INDICES.items():
        for aug_id in aug_list:
            finger_images = []
            complete = True
            sample_counter += 1
            print(f"\n📦 Subject {subject_id} — Image {img_num}_{aug_id} (Sample #{sample_counter:02d})")

            for finger in FINGER_NUMS:
                fname = f"{subject_id}_{finger}_{img_num}_{aug_id}_Augmented.png"
                img_path = os.path.join(subject_path, fname)
                print(f"🖼️ Loading: {img_path}")

                if not os.path.exists(img_path):
                    print(f"❌ File not found: {img_path}")
                    complete = False
                    break

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, IMAGE_SIZE)
                img_denoised = apply_denoising(img, h=10)
                img_eq = exposure.equalize_hist(img_denoised)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
                finger_images.append(img_norm)

            if complete and len(finger_images) == 6:
                fused = np.vstack(finger_images)
                fused_samples.append(fused)
                label = f"{subject_id}_fused_aug{sample_counter:02d}"
                labels.append(label)
                print(f"✅ Created fused sample: {label}")
            else:
                print(f"⚠️ Incomplete sample skipped for Subject {subject_id}, Sample #{sample_counter:02d}")

    return fused_samples, labels

# === LOAD TRAINING DATA ===
train_data = []
train_labels = []

print("🔄 Scanning dataset folders...")
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="📥 Loading Subjects"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔍 Processing subject: {subj}")
    samples, labels = fuse_fingers_p1(subject_path, subj)
    train_data.extend(samples)
    train_labels.extend(labels)

train_data = np.array(train_data)
train_labels = np.array(train_labels)

print("\n📊 ✅ Final Train Data Loaded")
print(f"   ➤ Total Samples: {train_data.shape[0]}")
print(f"   ➤ Image Shape: {train_data[0].shape}")
print(f"   ➤ Labels: {train_labels[:5]}")

# === 2DPCA FUNCTIONS ===
def compute_2dpca(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"   ➕ Added contribution from sample {i+1}")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ 2DPCA matrix W shape: {eig_vecs.shape}")
    return eig_vecs

def project_2dpca(images_2d, W):
    print("\n📐 Projecting images into 2DPCA space...")
    projected = []
    for i, img in enumerate(images_2d):
        feat = img @ W
        projected.append(feat)
        if i < 3:
            print(f"   🧮 Projected shape for sample {i+1}: {feat.shape}")
    return projected

def flatten_2dpca_features(projected_images):
    print("\n📦 Flattening projected features for classifier...")
    return np.array([img.flatten() for img in projected_images])

# === RUN 2DPCA ===
num_components = 47
W = compute_2dpca(train_data, num_components)
projected_features = project_2dpca(train_data, W)
flat_features = flatten_2dpca_features(projected_features)

print(f"\n✅ Final Feature Matrix Shape for Classification: {flat_features.shape}")


Test

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]


PROTOCOL2_TEST_FILES = [
    ("1", "3_Augmented"),("2", "3_Augmented"),("3", "3_Augmented"),("4", "3_Augmented"),
    ("1", ""), ("2", ""), ("3", ""), ("4", "")
]

test_data = []
test_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    return cv2.fastNlMeansDenoising(image, h=h)

# === FUNCTION TO LOAD & FUSE TEST FINGERS ===
def fuse_fingers_test(subject_path, subject_id):
    fused_samples = []
    labels = []

    for img_num, suffix in PROTOCOL2_TEST_FILES:
        fused_vector = []
        complete = True
        label_suffix = suffix if suffix else "orig"

        print(f"\n📦 Subject {subject_id} — Test Image {img_num}_{label_suffix}")

        for finger in FINGER_NUMS:
            fname = f"{subject_id}_{finger}_{img_num}_{suffix}.png" if suffix else f"{subject_id}_{finger}_{img_num}.png"
            img_path = os.path.join(subject_path, fname)
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"❌ File not found: {img_path}")
                complete = False
                break

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            img_denoised = apply_denoising(img, h=10)
            img_eq = exposure.equalize_hist(img_denoised)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            fused_vector.append(img_norm)

        if complete and len(fused_vector) == 6:
            fused_img = np.vstack(fused_vector)
            label = os.path.splitext(fname)[0].rsplit('_', 3)[0]  # Extract subject_finger style ID
            fused_samples.append(fused_img)
            labels.append(label)
            print(f"✅ Fused test sample saved as: {label}")
        else:
            print(f"⚠️ Incomplete fusion skipped: {subject_id} img {img_num}_{label_suffix}")

    return fused_samples, labels

# === LOAD TEST DATA FOR ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🧪 Loading Test Data"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue
    print(f"\n🔍 Processing subject: {subj}")
    samples, labels = fuse_fingers_test(subject_path, subj)
    test_data.extend(samples)
    test_labels.extend(labels)

test_data = np.array(test_data)
test_labels = np.array(test_labels)

print("\n📊 ✅ Final Test Data Loaded")
print(f"   ➤ Total Test Samples: {len(test_data)}")
print(f"   ➤ Example Test Image Shape: {test_data[0].shape}")
print(f"   ➤ Example Label: {test_labels[0]}")

# === PROJECT INTO 2DPCA SPACE ===
print("\n📐 Projecting Test Images into 2DPCA Space...")
proj_test_data = [img @ W for img in test_data]
flat_test_data = np.array([feat.flatten() for feat in proj_test_data])
print(f"✅ Final Flattened Test Feature Shape: {flat_test_data.shape}")


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(flat_test_data)

print("\n📤 Matching test samples using 2DPCA features (Person ID only)...")

# === Step 1: Compare each test sample to all training samples ===
for i in range(total_tests):
    test_vector = flat_test_data[i]
    true_label = test_labels[i]  # e.g., "0001_2_4_3_Augmented"

    # 📏 Compute Manhattan distances to all training vectors
    distances = np.sum(np.abs(flat_features - test_vector), axis=1)

    # 🏆 Nearest neighbor index
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]

    # 🎯 Extract only subject IDs
    true_id = true_label.split("_")[0]     # e.g., "0001"
    pred_id = predicted_label.split("_")[0]

    # ✅ Person ID match
    if pred_id == true_id:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Person Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
